<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 La Línea que Mejor Pasa por los Puntos: Abono y Cosecha 📈
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 04
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/04%20-%20No%20Parametrica%20Regresion%20y%20Etica/Para%20Dummies/01_Regresion_Simple_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no matemáticos»** del [cuaderno 01 del Módulo 04 (Regresión lineal simple y correlación)](../01_Regresion_Lineal_Simple_y_Correlacion.ipynb). Un agricultor te pregunta: «**si le echo más abono, ¿cuánto más cosecho?**». Dibujas cada parcela como un punto (abono hacia la derecha, cosecha hacia arriba) y buscas **la línea que mejor resume la nube**. Esa línea te sirve para **explicar** y para **predecir**, pero también puede engañarte.

| Lo que decimos | Lo que significa en la vida diaria |
|---|---|
| **Línea de ajuste** | La recta que pasa «lo más cerca posible» de todos los puntos a la vez |
| **Pendiente** | Cuánto sube la cosecha, en promedio, por cada kilo más de abono |
| **Error de la línea** | Qué tan lejos queda cada parcela de la línea (se suman todos; los más lejanos pesan más) |
| **Correlación** | Un número entre −1 y 1 que dice qué tan **parejo** suben o bajan dos cosas **en línea recta** |

Al terminar podrás explicar:
1. Cómo se **escoge** la mejor línea (la que se equivoca menos en total) y cómo se **lee** su pendiente.
2. Por qué **un número de correlación solo no basta** (una curva perfecta puede dar casi cero, y un solo punto raro puede inflarlo).
3. Por qué **no hay que pasarse del rango** de los datos al predecir.

> 🔗 Si luego quieres ver la versión **con intervalos, residuos y simulaciones**: [cuaderno estándar](../01_Regresion_Lineal_Simple_y_Correlacion.ipynb).

---
## 1. Buscar la Línea que Menos se Equivoca 🔍

Medimos el **abono** (kg por hectárea) y la **cosecha** (toneladas por hectárea) en **30 parcelas** (datos **inventados**). Probamos tres líneas «a ojo» y contamos cuánto se equivoca cada una en total; después dejamos que el computador encuentre la mejor.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)                      # semilla fija: siempre salen los mismos numeros
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})

abono = rng.uniform(20, 140, 30)                                   # kg de abono por hectarea
cosecha = 12 + 0.07 * abono + rng.normal(0, 2, 30)                 # t/ha: sube con el abono, pero con "ruido" (clima, suelo...)

def error_total(intercepto, pendiente):
    """Suma de los errores al cuadrado: lo lejano pesa mucho mas que lo cercano."""
    return ((cosecha - (intercepto + pendiente * abono)) ** 2).sum()

intentos = {"Linea 1 (casi plana)": (16.0, 0.02), "Linea 2 (muy empinada)": (5.0, 0.15), "Linea 3 (a ojo)": (13.0, 0.06)}
for nombre, (b0, b1) in intentos.items():
    print(f"{nombre:24s}: error total = {error_total(b0, b1):7.1f}")

pendiente, intercepto = np.polyfit(abono, cosecha, 1)              # la mejor linea (la que menos se equivoca en total)
print(f"{'Mejor linea (polyfit)':24s}: error total = {error_total(intercepto, pendiente):7.1f}  -> cosecha = {intercepto:.2f} + {pendiente:.3f} x abono")
print(f"Lectura: por cada 10 kg mas de abono, la cosecha sube en promedio {10 * pendiente:.2f} t/ha.")

plt.figure(figsize=(9, 4.2))
plt.scatter(abono, cosecha, color="#2563eb", label="parcelas (inventadas)")
xs = np.array([20, 140])
for (nombre, (b0, b1)), c in zip(intentos.items(), ["#94a3b8", "#f59e0b", "#16a34a"]): plt.plot(xs, b0 + b1 * xs, color=c, ls="--", label=nombre)
plt.plot(xs, intercepto + pendiente * xs, color="black", lw=2, label="mejor linea")
plt.xlabel("abono (kg/ha)"); plt.ylabel("cosecha (t/ha)"); plt.title("De todas las lineas, una se equivoca menos"); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

assert all(error_total(intercepto, pendiente) <= error_total(b0, b1) for b0, b1 in intentos.values())
assert 0.03 < pendiente < 0.12

---
**Lección:** la línea «casi plana» se equivoca en total **167.7**, la «muy empinada» **325.5** y la «a ojo» **77.3**; la mejor línea que encuentra el computador se equivoca **74.9** (siempre gana, porque se define como la que menos se equivoca). Su lectura: **cosecha = 12.44 + 0.068 × abono**, es decir, **por cada 10 kg más de abono, la cosecha sube en promedio 0.68 t/ha** (en estos datos inventados la verdad era 12 + 0.07 × abono). Ojo: es un **promedio**; ninguna parcela cae exactamente sobre la línea.

---
## 2. La Correlación: Un Número que Puede Engañar 🎭

La **correlación** resume en un número entre −1 y 1 qué tan parejo suben (o bajan) dos cosas **en línea recta**. Cerca de 1: suben juntas; cerca de −1: una sube cuando la otra baja; **cerca de 0: no hay relación en línea recta** (¡pero puede haber una curva!). Veamos tres nubes de puntos.

In [ ]:
g = np.random.default_rng(6)
# Nube 1: relacion en linea recta con ruido
x1 = g.uniform(0, 10, 40); y1 = 2 * x1 + g.normal(0, 3, 40)
# Nube 2: relacion PERFECTA pero en curva (el dano baja y luego sube segun el abono)
x2 = np.linspace(-5, 5, 41); y2 = x2 ** 2
# Nube 3: dos cosas SIN relacion, mas UN solo punto muy raro (un error al anotar)
x3 = g.uniform(0, 10, 39); y3 = g.uniform(0, 10, 39)
x3b, y3b = np.r_[x3, 60.0], np.r_[y3, 60.0]

correlacion = lambda a, b: np.corrcoef(a, b)[0, 1]
r1, r2, r3, r3b = correlacion(x1, y1), correlacion(x2, y2), correlacion(x3, y3), correlacion(x3b, y3b)
print(f"Nube 1 (recta con ruido)         : correlacion = {r1:5.2f}")
print(f"Nube 2 (curva perfecta)          : correlacion = {r2:5.2f}   <- casi cero, aunque y depende TOTALMENTE de x")
print(f"Nube 3 (sin relacion)            : correlacion = {r3:5.2f}")
print(f"Nube 3 + un solo punto raro      : correlacion = {r3b:5.2f}   <- un punto la dispara")

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].scatter(x1, y1, color="#2563eb"); ax[0].set_title(f"Recta con ruido (r = {r1:.2f})")
ax[1].scatter(x2, y2, color="#16a34a"); ax[1].set_title(f"Curva perfecta (r = {r2:.2f})")
ax[2].scatter(x3b[:-1], y3b[:-1], color="#94a3b8"); ax[2].scatter(x3b[-1:], y3b[-1:], color="#dc2626", s=80, marker="X"); ax[2].set_title(f"Un punto raro (r = {r3b:.2f})")
plt.tight_layout(); plt.show()

assert r1 > 0.7 and abs(r2) < 0.1 and abs(r3) < 0.35 and r3b > 0.8

---
##### 🛠️ Práctica 1: Predecir con la línea... y saber cuánto se equivoca

Con la mejor línea de la sección 1 (`intercepto`, `pendiente`) y los arreglos `abono` y `cosecha`:

1. Predice la cosecha para **80 kg/ha**: `pred_80 = intercepto + pendiente * 80`.
2. Calcula las distancias de cada parcela a la línea: `errores = cosecha - (intercepto + pendiente * abono)`, y su tamaño típico `error_tipico = np.std(errores)`.
3. Comprueba con `assert` que `pred_80` está entre 15 y 20 t/ha y que `error_tipico` está entre 1 y 3.
4. Comenta con tus palabras: si el agricultor pregunta por **su** parcela, ¿le darías solo `pred_80` o también un rango como `pred_80 ± 2 × error_tipico`?

In [ ]:
# Escribe tu código aquí

# pred_80 = ...
# errores = ...
# error_tipico = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
pred_80 = intercepto + pendiente * 80
errores = cosecha - (intercepto + pendiente * abono)
error_tipico = np.std(errores)
print(f"Prediccion para 80 kg/ha: {pred_80:.1f} t/ha | error tipico de la linea: {error_tipico:.1f} t/ha | rango razonable: {pred_80 - 2 * error_tipico:.1f} a {pred_80 + 2 * error_tipico:.1f} t/ha")

assert 15 < pred_80 < 20 and 1 < error_tipico < 3
print("Hay que dar el rango, no solo el numero: la linea es un promedio y cada parcela se desvia de ella.")
```
</details>

---

---
**Lección:** la nube en línea recta da una correlación de **0.91**; la **curva perfecta** (donde y depende por completo de x) da **0.00**, porque solo se mide el parentesco en línea recta; y dos cosas **sin relación** dan −0.02, pero si agregas **un solo punto raro** (por ejemplo, un error al anotar) la correlación salta a **0.88**. Por eso siempre se **dibuja** antes de resumir con un número. Y recuerda: aunque la correlación sea alta, eso **no** prueba que una cosa cause la otra.

---
## 3. No Te Pases del Rango: el Peligro de Adivinar Más Allá 🚧

Una planta no responde al abono **para siempre**: a partir de cierto punto ya no rinde más. Si ajustas la línea con parcelas de **20 a 100 kg/ha** y luego la usas para predecir con **300 kg/ha**, la línea sigue subiendo aunque la planta ya no pueda más. Probemos con una respuesta **inventada** que se «aplana».

In [ ]:
real = lambda abono_: 20 * (1 - np.exp(-abono_ / 40))               # respuesta inventada: se aplana cerca de 20 t/ha
g = np.random.default_rng(5)
ab = g.uniform(20, 100, 40); co = real(ab) + g.normal(0, 0.6, 40)
m, b = np.polyfit(ab, co, 1)

tabla = pd.DataFrame({"abono (kg/ha)": [60, 100, 150, 300]})
tabla["la linea predice (t/ha)"] = b + m * tabla["abono (kg/ha)"]
tabla["lo que pasa de verdad (t/ha)"] = real(tabla["abono (kg/ha)"])
tabla["error (t/ha)"] = tabla["la linea predice (t/ha)"] - tabla["lo que pasa de verdad (t/ha)"]
print(tabla.round(1).to_string(index=False))

plt.figure(figsize=(9, 4))
xx = np.linspace(0, 320, 200)
plt.scatter(ab, co, color="#2563eb", label="parcelas medidas (20-100 kg)"); plt.plot(xx, real(xx), color="#16a34a", label="lo que pasa de verdad")
plt.plot(xx, b + m * xx, color="#dc2626", ls="--", label="la linea ajustada"); plt.axvspan(100, 320, color="grey", alpha=0.12, label="fuera de lo medido")
plt.xlabel("abono (kg/ha)"); plt.ylabel("cosecha (t/ha)"); plt.title("Fuera del rango, la linea promete lo imposible"); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

assert abs(tabla["error (t/ha)"].iloc[0]) < 1.5 and tabla["error (t/ha)"].iloc[3] > 15

---
**Lección:** dentro de lo medido (60 y 100 kg) la línea se equivoca poco (−0.9 y +1.4 t/ha). Pero a 150 kg ya se pasa por **6.7 t/ha** y a 300 kg predice **45.6 t/ha** cuando la planta, que se aplana, da apenas **20.0 t/ha**: un error de **25.6 t/ha**. La línea no «sabe» que la planta tiene un tope; solo repite lo que vio. Moraleja: **predice dentro del rango de tus datos**, y si necesitas ir más allá, consigue datos de ese rango.

---
## Resumen en una frase 🎯

> **La línea que mejor pasa por los puntos es la que menos se equivoca en total y su pendiente dice cuánto cambia la cosecha por cada kilo más de abono en promedio, pero un número de correlación nunca reemplaza mirar el gráfico, cada parcela se desvía de la línea (por eso se da un rango) y no hay que usar la línea fuera de los datos medidos.**

### Lo que aprendiste hoy:
- ✅ **Escoger la mejor línea** comparando cuánto se equivoca cada una, y **leer su pendiente** («por cada 10 kg más, +X t/ha»).
- ✅ **La correlación** mide solo asociación en línea recta: una curva perfecta puede dar casi 0 y **un solo punto raro** puede inflarla.
- ✅ **Predecir con rango**, no con un solo número, y **no pasarse** de lo que se midió.

### 🧠 ¿Ya lo tienes claro?

1. Alguien dice: «la correlación entre el número de bomberos en un incendio y el daño es alta, así que los bomberos causan daño». ¿Dónde está el error? ¿Qué otra cosa explica las dos?
2. Una nube de puntos da correlación casi cero. ¿Puedes asegurar que las dos cosas no están relacionadas? ¿Qué mirarías antes de decidir?
3. Ajustaste una línea con parcelas de 20 a 100 kg de abono y te piden la cosecha con 400 kg. ¿Qué respondes?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos (Edición Didáctica Para Dummies)</i>
  </p>
</div>